<a href="https://colab.research.google.com/github/Snickmax/CrudEmmbedding/blob/main/CrudEmbedding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install chromadb langchain sentence-transformers numpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 5.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 606.2/606.2 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.6/278.6 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.8/94.8 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 71.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.2/93.2 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 83.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.9/54.9 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 5.0 MB/s eta 0:00:00


In [ ]:
!pip install -U langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 kB 4.5 MB/s eta 0:00:00


In [ ]:
# Notebook: Operaciones CRUD con ChromaDB y Embeddings

# Importar librerías necesarias
from langchain.vectorstores import Chroma
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.schema import Document
import os

# Configurar el modelo de embeddings y la base de datos
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

class DocumentManager:
    def __init__(self, directorio="chroma_db"):
        self.directorio = directorio
        self.base_datos = self.configurar_base_datos(directorio)

    def configurar_base_datos(self, directorio):
        os.makedirs(directorio, exist_ok=True)
        base_datos = Chroma(persist_directory=directorio, embedding_function=embedding_model)
        print(f"Base de datos configurada en: {directorio}")
        return base_datos

    # Crear un documento en la base de datos con un ID único
    def agregar_documento(self, texto):
        # Generar un ID único para el documento
        id_documento = str(hash(texto))  # Usamos el hash del texto como ID
        documento = Document(page_content=texto, metadata={"id": id_documento})
        self.base_datos.add_documents([documento])
        print(f"Documento agregado: '{texto}' con ID {id_documento}")
        return id_documento  # Devolvemos el ID para futuras referencias

    # Leer documentos mediante búsqueda de similitud
    def consultar_documentos(self, consulta, k=5):
        resultados = self.base_datos.similarity_search(consulta, k=k)
        return [(resultado.page_content, resultado.metadata) for resultado in resultados]

    # Actualizar un documento
    def actualizar_documento(self, texto_viejo, texto_nuevo):
        # Realizamos una búsqueda de similitud para encontrar el documento
        resultados = self.base_datos.similarity_search(texto_viejo, k=1)  # Buscamos el documento más similar

        if resultados:
            # Si encontramos el documento, obtenemos su ID
            documento_antiguo = resultados[0]
            id_documento = documento_antiguo.metadata.get("id")

            if id_documento:
                # Eliminar el documento antiguo utilizando su ID
                self.base_datos.delete([id_documento])  # Eliminamos solo con el id
                print(f"Documento eliminado: '{texto_viejo}'")

                # Agregar el nuevo documento
                self.agregar_documento(texto_nuevo)
                print(f"Documento actualizado: '{texto_viejo}' a '{texto_nuevo}'")
            else:
                print(f"Error: El documento con el texto '{texto_viejo}' no tiene un ID válido para eliminar.")
        else:
            print(f"No se encontró el documento con el texto: '{texto_viejo}'")

    # Eliminar un documento
    def eliminar_documento(self, texto):
        # Realizamos una búsqueda de similitud para encontrar el documento
        resultados = self.base_datos.similarity_search(texto, k=1)  # Buscamos el documento más similar

        if resultados:
            # Si encontramos el documento, obtenemos su ID
            documento = resultados[0]
            id_documento = documento.metadata.get("id")

            if id_documento:
                # Eliminar el documento utilizando su ID
                self.base_datos.delete([id_documento])
                print(f"Documento eliminado: '{texto}'")
            else:
                print(f"Error: El documento con el texto '{texto}' no tiene un ID válido para eliminar.")
        else:
            print(f"No se encontró el documento con el texto: '{texto}'")

# Uso del notebook
if __name__ == "__main__":
    # Crear un objeto DocumentManager para gestionar los documentos
    doc_manager = DocumentManager(directorio="chroma_db")

    # Agregar documentos
    doc_manager.agregar_documento("Este es un texto de ejemplo.")
    doc_manager.agregar_documento("Otro texto para pruebas.")

    # Consultar documentos
    consulta = "texto de ejemplo"
    resultados = doc_manager.consultar_documentos(consulta)
    print("Resultados de la consulta:", resultados)

    # Actualizar un documento
    doc_manager.actualizar_documento("Este es un texto de ejemplo.", "Texto de ejemplo actualizado.")

    # Eliminar un documento
    doc_manager.eliminar_documento("Texto de ejemplo actualizado.")


<ipython-input-3-d6010c61ec34>:10: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.7k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

<ipython-input-3-d6010c61ec34>:19: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  base_datos = Chroma(persist_directory=directorio, embedding_function=embedding_model)


Base de datos configurada en: chroma_db


Documento agregado: 'Este es un texto de ejemplo.' con ID -8808639272667882610
Documento agregado: 'Otro texto para pruebas.' con ID -2731942575484916397
Resultados de la consulta: [('Este es un texto de ejemplo.', {'id': '-8808639272667882610'}), ('Otro texto para pruebas.', {'id': '-2731942575484916397'})]
Documento eliminado: 'Este es un texto de ejemplo.'
Documento agregado: 'Texto de ejemplo actualizado.' con ID 2996496253532104409
Documento actualizado: 'Este es un texto de ejemplo.' a 'Texto de ejemplo actualizado.'
Documento eliminado: 'Texto de ejemplo actualizado.'
